# setup

In [1]:
import sys

sys.path.append("../")

In [2]:
import json
import os
from typing import *

import dotenv
import pandas
from tqdm.notebook import tqdm

dotenv.load_dotenv()

True

In [3]:
from logging import Logger, StreamHandler, basicConfig

basicConfig(level="INFO", format="%(asctime)s - %(name)s - %(levelname)s - %(message)s")
logger = Logger("notebook")
logger.addHandler(StreamHandler(sys.stdout))

logger.info("Hello, world!")

Hello, world!


In [4]:
%load_ext autoreload
%autoreload 2

# mysql

In [5]:
from crunch_titles._database import Database

if "database" in globals():
    database.__exit__(None, None, None) # type: ignore

database = Database(
    host=os.environ["DATABASE_HOST"],
    user=os.environ["DATABASE_USER"],
    password=os.environ["DATABASE_PASSWORD"],
    account_service_name=os.environ["ACCOUNT_DATABASE_NAME"],
    competition_service_name=os.environ["COMPETITION_DATABASE_NAME"],
    enable_caching=False,
)

database.__enter__()

In [6]:
from crunch_titles._repository import LoadEverythingRepository

repository = LoadEverythingRepository(
    database=database,
    logger=logger,
)

  0%|          | 0/13 [00:00<?, ?method/s]

load: competitions
load: users
load: leaderboard_definitions
load: targets
load: rounds
load: phases
load: crunches
load: crunch_targets
load: leaderboards
load: positions
load: paid_checkpoint_payouts
load: title_leaderboards
load: title_positions
len: _competitions=19
len: _competition_by_id=19
len: _competition_by_name=19
len: _user_by_id=2866
len: _user_by_login=2866
len: _default_leaderboard_definition_by_competition_id=21
len: _usable_targets_by_competition_id=22
len: _rounds_by_competition_id=21
len: _out_of_sample_phase_by_round_id=126
len: _published_crunch_by_phase_id=185
len: _crunch_target_by_crunch_id_and_target_id=954
len: _leaderboard_by_crunch_target_id_and_leaderboard_definition_id=712
len: _position_by_leaderboard_id=712
len: _payouts_by_competition_id=3
len: _payout_recipients_by_payout_id=45
len: _title_leaderboard_by_competition_id_and_year=0
len: _title_positions_by_leaderboard_id=0


# querying common data

In [7]:
competitions = repository.find_all_competitions()
competitions.sort(key=lambda x: x["name"] + " ")

for competition in competitions:
    print(competition["name"])

adialab
broad-1
broad-2
broad-3
broad-obesity-1
broad-obesity-2
broad-obesity-3
causality-discovery
datacrunch
datacrunch-2
datacrunch-rally
falcon
mid-one
numinous
structural-break
structural-break-open-benchmark
structural-break-real-time
synth
venture-capital-portfolio-prediction


In [8]:
structural_break = next(filter(lambda x: x["name"] == "structural-break", competitions))
structural_break_open_benchmark = next(filter(lambda x: x["name"] == "structural-break-open-benchmark", competitions), None)
datacrunch = next(filter(lambda x: x["name"] == "datacrunch", competitions))
falcon = next(filter(lambda x: x["name"] == "falcon", competitions))
structural_break2 = next(filter(lambda x: x["name"] == "structural-break-open-benchmark", competitions), None)

# compute positions

## compute positions from leaderboards

In [9]:
from crunch_titles._model import Competition
from crunch_titles._position import LeaderboardPosition, determine_positions

grouped_positions: List[Tuple[Tuple[Competition, int], List[LeaderboardPosition]]] = []

for competition in tqdm(competitions):
# for competition in tqdm([datacrunch]):
    grouped_positions.extend(determine_positions(
        competition=competition,
        repository=repository,
    ))

  0%|          | 0/19 [00:00<?, ?it/s]

### debug

In [10]:
from crunch_titles._debug import print_competition_positions_count

print_competition_positions_count(
    print,
    grouped_positions
)

competition                                        total    unique
adialab:0                                             375        1
broad-1:0                                              56        1
broad-2:0                                              16        1
broad-3:0                                              15        1
broad-obesity-1:0                                      67        1
broad-obesity-2:0                                      72        1
causality-discovery:0                                 207        1
datacrunch:2024                                      2101       26
datacrunch:2025                                      6380       50
datacrunch-rally:0                                     47        1
falcon:2025                                            96        6
falcon:2026                                           134        7
mid-one:2024                                           59        1
mid-one:2025                                           63     

## filter and average into a single point if necessary

In [11]:
from crunch_titles import LocalTitlePositionPerCompetitionYearList, average_leaderboards

averaged_leaderboards: LocalTitlePositionPerCompetitionYearList = []

for (competition, year), positions in grouped_positions:
    title_positions, user_count = average_leaderboards(
        competition=competition,
        year=year,
        positions=positions,
    )

    if len(title_positions):
        averaged_leaderboards.append(((competition, year), title_positions, user_count))

## merge

(only for structural-break/open-benchmark)

In [12]:
from crunch_titles import merge_leaderboards

merged_leaderboards = merge_leaderboards(
    repository=repository,
    averaged_leaderboards=averaged_leaderboards,
)

### debug

In [13]:
for (competition, year), positions, user_count in merged_leaderboards:
    if competition != datacrunch:
        continue

    print(competition["name"], year, len(positions), user_count)
    
    positions = list(positions)
    positions.sort(key=lambda x: x["rank"])

    for position in positions:
        print(f"{position["user"]["login"]:<30}  {position["rank"]:<3}")

datacrunch 2024 96 122
right-pachara                   1  
selective-zhu                   2  
spicy-questo                    3  
tarandros                       3  
amused-david                    3  
vast-chananyu                   4  
grubby-borit                    5  
crowded-anirban                 6  
trottefox                       7  
giras                           7  
semantic-hu                     7  
soft-eduard                     8  
video-taped-grzegorz            8  
noob-saibot                     9  
kain                            10 
kat                             10 
coolplay                        11 
philosophical-ronny             11 
bitter-krittanut                11 
faithful-puretwin               11 
ghiffaryr                       12 
lcrm                            12 
ltamas97                        12 
cold-nasirudeen                 13 
multiple-t                      13 
substantial-alex                13 
historic-joe                    13 
duusc

In [14]:
from crunch_titles._debug import print_user_ranks

print_user_ranks(print, merged_leaderboards, "tarandros")
print("")
print_user_ranks(print, merged_leaderboards, "cyber-bob")

user                 competition:year                                     rank   usr.cnt
tarandros            adialab:0                                             1.0       375
tarandros            broad-1:0                                             4.0        56
tarandros            broad-2:0                                             1.0        16
tarandros            broad-3:0                                             5.0        15
tarandros            broad-obesity-1:0                                    15.0        67
tarandros            broad-obesity-2:0                                    28.0        72
tarandros            causality-discovery:0                                 2.0       207
tarandros            datacrunch:2024                                       3.0       122
tarandros            datacrunch:2025                                      11.0       218
tarandros            datacrunch-rally:0                                   42.0        47
tarandros            

## compute medals

In [15]:
from crunch_titles import distribute_medals

for (competition, year), positions, user_count in merged_leaderboards:
    distribute_medals(
        positions=positions,
        user_count=user_count,
    )

In [16]:
from crunch_titles._debug import print_medals

print_medals(
    print,
    merged_leaderboards,
    "structural-break",
    2024,
)

login                                       rank  medal 
brandao                                        1  GOLD  
mario-filho                                    1  GOLD  
rafael                                         1  GOLD  
joao-peinado                                   1  GOLD  
farukcan-saglam                                2  SILVER
abishek                                        2  SILVER
lcrm                                           3  BRONZE
mirko                                          3  BRONZE
valuable-j                                     4  TOP_10_PERCENT
tuah-jihan                                     4  TOP_10_PERCENT
mutian-hong                                    5  TOP_10_PERCENT
guoqin-gu                                      5  TOP_10_PERCENT
giras                                          5  TOP_10_PERCENT
leo-nguyen                                     6  TOP_10_PERCENT
anxious-james                                  6  TOP_10_PERCENT
condor                          

In [17]:
from crunch_titles import count_medals_per_user

medal_counts = count_medals_per_user(
    merged_leaderboards=merged_leaderboards,
)

In [18]:
from crunch_titles._debug import print_medal_counts

print_medal_counts(print, medal_counts)

user                             gold silver bronze top 10%
tarandros                           2      1      3      4
trottefox                           2      -      -      3
plutos-farm                         2      -      -      1
multiple-t                          1      2      -      1
kalin-nonchev                       1      1      1      -
anxious-james                       1      1      -      3
smoggy-mahcih                       1      1      -      1
mutian-hong                         1      -      1      1
coolplay                            1      -      -      3
fwdscttr                            1      -      -      -
agreeable-amine                     1      -      -      -
donald-armstrong                    1      -      -      -
many-frank                          1      -      -      1
marcfm                              1      -      -      -
short-niko                          1      -      -      -
cute-air                            1      -      -    

## compute titles

### grandmasters

In [19]:
from crunch_titles._title import compute_grandmasters

grandmaster_user_ids = compute_grandmasters(
    medal_counts=medal_counts,
)

len(grandmaster_user_ids)

11

### masters

In [20]:
from crunch_titles._title import compute_masters

master_user_ids = compute_masters(
    medal_counts=medal_counts, 
    grandmasters=grandmaster_user_ids,
)

len(master_user_ids)

26

### experts

In [21]:
from crunch_titles._title import compute_experts

expert_user_ids = compute_experts(
    medal_counts=medal_counts,
    grandmasters=grandmaster_user_ids,
    masters=master_user_ids,
)

len(expert_user_ids)

34

### ranked

In [22]:
from crunch_titles._title import compute_ranked

ranked_user_ids = compute_ranked(
    medal_counts=medal_counts,
    grandmasters=grandmaster_user_ids,
    masters=master_user_ids,
    experts=expert_user_ids,
)

len(ranked_user_ids)

189

### remaining

In [23]:
from crunch_titles._title import compute_titles

remaining_user_ids = compute_titles(
    medal_counts=medal_counts,
    predicate=lambda _: True,
    other_sets=[grandmaster_user_ids, master_user_ids, expert_user_ids, ranked_user_ids]
)

for user, medal_count in medal_counts:
    if user["id"] in remaining_user_ids:
        print(user["login"], medal_count)

assert len(remaining_user_ids) == 0

### statistics

In [24]:
from crunch_titles._debug import print_titles_count

print_titles_count(
    print,
    grandmaster_user_ids,
    master_user_ids,
    expert_user_ids,
    ranked_user_ids,
)

Titles count:
- grandmasters 11
- masters 26
- experts 34
- ranked 189
- total 260


In [25]:
_dfs: List[pandas.DataFrame] = []

for (competition, year), positions, user_count in merged_leaderboards:
    _dfs.append(pandas.DataFrame([
        {
            "competition": f"{competition['name']}:{year}" if year else competition['name'],
            "user": f"{position['user']['id']}.{position['user']['login']}",
            "average": position["average"],
            "participation_count": position["participation_count"],
            "rank": position["rank"],
        }
        for position in positions
    ]))

merged_leaderboards_df = pandas.concat(_dfs, ignore_index=True)
merged_leaderboards_df

,competition,user,average,participation_count,rank
0,adialab,3048.tarandros,1.0,1,1
1,adialab,3738.fun-robert,2.0,1,2
2,adialab,2281.hj67,3.0,1,3
3,adialab,3623.casual-andy,4.0,1,4
4,adialab,2275.cautious-mayeul,5.0,1,5
...,...,...,...,...,...
1947,venture-capital-portfolio-prediction,3529.elaborate-zoe,54.0,1,54
1948,venture-capital-portfolio-prediction,2281.hj67,55.0,1,55
1949,venture-capital-portfolio-prediction,1637.historic-joe,56.0,1,56
1950,venture-capital-portfolio-prediction,4777.capable-jiaming,57.0,1,57


In [26]:
pivoted_leaderboards_df = merged_leaderboards_df.pivot(index="user", columns="competition", values="rank")

pivoted_leaderboards_df["gold"] = 0
pivoted_leaderboards_df["silver"] = 0
pivoted_leaderboards_df["bronze"] = 0
pivoted_leaderboards_df["top_10%"] = 0
pivoted_leaderboards_df["top_20%"] = 0

for user, medal_count in medal_counts:
    user_key = f"{user['id']}.{user['login']}"
    pivoted_leaderboards_df.loc[user_key, "gold"] = medal_count.get("GOLD", 0)
    pivoted_leaderboards_df.loc[user_key, "silver"] = medal_count.get("SILVER", 0)
    pivoted_leaderboards_df.loc[user_key, "bronze"] = medal_count.get("BRONZE", 0)
    pivoted_leaderboards_df.loc[user_key, "top_10%"] = medal_count.get("TOP_10_PERCENT", 0)
    pivoted_leaderboards_df.loc[user_key, "top_20%"] = medal_count.get("TOP_20_PERCENT", 0)

pivoted_leaderboards_df.sort_values(by=["gold", "silver", "bronze"], ascending=False, inplace=True)
pivoted_leaderboards_df.to_csv("meta_ranking.csv")

pivoted_leaderboards_df

competition,adialab,broad-1,broad-2,broad-3,broad-obesity-1,broad-obesity-2,causality-discovery,datacrunch-rally,datacrunch:2024,datacrunch:2025,...,mid-one:2024,mid-one:2025,numinous:2026,structural-break,venture-capital-portfolio-prediction,gold,silver,bronze,top_10%,top_20%
user,,,,,,,,,,,,,,,,,,,,,
3048.tarandros,1.0,4.0,1.0,5.0,15.0,28.0,2.0,42.0,3.0,11.0,...,4.0,3.0,NaN,8.0,3.0,2,1,3,4,0
6710.trottefox,NaN,NaN,NaN,NaN,1.0,44.0,21.0,NaN,7.0,1.0,...,NaN,20.0,NaN,39.0,NaN,2,0,0,3,0
9193.plutos-farm,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,13.0,NaN,2,0,0,1,0
1529.multiple-t,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13.0,23.0,...,27.0,29.0,1.0,NaN,15.0,1,2,0,1,0
7180.kalin-nonchev,NaN,1.0,2.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,1,1,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9972.welcome-aditya,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,478.0,NaN,0,0,0,0,0
9974.okay-nariman,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,199.0,NaN,0,0,0,0,0
9977.purple-n,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,172.0,NaN,0,0,0,0,0


# compute full

In [ ]:
from crunch_titles import compute

compute(
    database=database,
    # competition_name=datacrunch["name"],
    competition_name=structural_break["name"],
    # competition_name=structural_break_open_benchmark["name"],
    logger=logger,
)

database.commit()

In [ ]:
from crunch_titles import compute

for competition in competitions:
    compute(
        database=database,
        competition_name=competition["name"],
        logger=logger,
    )

database.commit()

In [28]:
logger.warning("hello")

hello


# insert in database

## cleanup

In [49]:
database.competition.insert("SET FOREIGN_KEY_CHECKS = 0;")
database.competition.insert("TRUNCATE `title_leaderboards`;")
database.competition.insert("TRUNCATE `title_positions`;")
database.competition.insert("UPDATE `users` SET `title` = 'NOVICE';")
database.competition.insert("SET FOREIGN_KEY_CHECKS = 1;")

0

In [50]:
database.commit()

## title positions

In [51]:
for (competition, year), _ in grouped_positions:
    repository.delete_title_leaderboard_by_competition_and_year(competition, year)

for (competition, year), title_positions, user_count in merged_leaderboards:
    title_leaderboard = repository.create_title_leaderboard({
        "competition_id": competition["id"],
        "year": year,
        "week_count": 0,  # TODO!
        "original_size": user_count,
        "size": len(title_positions),
    })

    for position in title_positions:
        repository.create_title_position({
            "leaderboard_id": title_leaderboard["id"],
            "user_id": position["user"]["id"],
            "averaged_rank": position["average"],
            "participation_count": position["participation_count"],
            "meta_rank": position["rank"],
            "medal": position["medal"],
        })

In [52]:
database.commit()

## user's titles

In [59]:
repository.set_title_for_users("GRANDMASTER", grandmaster_user_ids)
repository.set_title_for_users("MASTER", master_user_ids)
repository.set_title_for_users("EXPERT", expert_user_ids)
repository.set_title_for_users("RANKED", ranked_user_ids)

In [60]:
database.commit()

In [58]:
repository.set_title_for_users("RANKED", {1})
# repository.set_title_for_users("GRANDMASTER", {1})
database.commit()